In [1]:
# ============================================================
# MVTec 4-class counterfactual evaluation
# Surrogate consistency + real PatchCore confirmation
# Paper-ready plots and tables
# ============================================================

from pathlib import Path
import json
import time
import zlib
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageFilter

from catboost import CatBoostRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

CWD = Path.cwd()
print("Current working directory:")
print(CWD)

# ------------------------------------------------------------
# Expected filenames from previous MVTec notebooks
# ------------------------------------------------------------

DICE_CFS_FILENAME = "mvtec_dice_cfs.csv"
ALL_METHODS_CFS_FILENAME = "mvtec_all_methods_cfs.csv"

# For paper figures, prefer the all-method table when it exists.
# Set to False if you explicitly want to evaluate only the DiCE file.
PREFER_ALL_METHODS_CFS = True

CFS_META_FILENAME = "mvtec_dice_cfs_metadata.json"
SURROGATE_MODEL_FILENAME = "mvtec_surrogate_catboost.cbm"
SURROGATE_META_FILENAME = "mvtec_surrogate_metadata.json"

# ------------------------------------------------------------
# Robust root detection
# ------------------------------------------------------------

candidate_roots = [
    CWD,
    CWD / "hpo_mvtec_4class",
    CWD.parent,
    CWD.parent / "hpo_mvtec_4class",
]

for base in [CWD, CWD.parent]:
    try:
        for p in base.rglob("hpo_mvtec_4class"):
            candidate_roots.append(p)
    except Exception:
        pass

seen = set()
candidate_roots_unique = []
for p in candidate_roots:
    p = p.resolve()
    if p not in seen:
        seen.add(p)
        candidate_roots_unique.append(p)

root_candidates = []

print("\nCandidate MVTec roots:")
for root in candidate_roots_unique:
    cfs_dir = root / "surrogate_pkl_cfs_metadata" / "mvtec_counterfactuals"
    dice_cfs_path = cfs_dir / DICE_CFS_FILENAME
    all_methods_cfs_path = cfs_dir / ALL_METHODS_CFS_FILENAME
    model_path = root / "surrogate_pkl_cfs_metadata" / SURROGATE_MODEL_FILENAME
    splits_dir = root / "mvtec_splits"

    ok = dice_cfs_path.exists() or all_methods_cfs_path.exists() or model_path.exists() or splits_dir.exists()
    print(f"- ok={ok:<5} | {root}")

    if ok:
        root_candidates.append(root)

if not root_candidates:
    raise FileNotFoundError(
        "Could not find hpo_mvtec_4class root. "
        "Run this notebook from the project root or from hpo_mvtec_4class."
    )

# Prefer the root that contains the selected CF file.
ROOT = None
for root in root_candidates:
    cfs_dir = root / "surrogate_pkl_cfs_metadata" / "mvtec_counterfactuals"
    preferred = cfs_dir / (ALL_METHODS_CFS_FILENAME if PREFER_ALL_METHODS_CFS else DICE_CFS_FILENAME)
    fallback = cfs_dir / DICE_CFS_FILENAME
    if preferred.exists() or fallback.exists():
        ROOT = root
        break

if ROOT is None:
    ROOT = root_candidates[0]

ART_DIR = ROOT / "surrogate_pkl_cfs_metadata"
CFS_DIR = ART_DIR / "mvtec_counterfactuals"
SPLITS_DIR = ROOT / "mvtec_splits"

preferred_cfs = CFS_DIR / (ALL_METHODS_CFS_FILENAME if PREFER_ALL_METHODS_CFS else DICE_CFS_FILENAME)
fallback_cfs = CFS_DIR / DICE_CFS_FILENAME

if preferred_cfs.exists():
    CFS_PATH = preferred_cfs
elif fallback_cfs.exists():
    CFS_PATH = fallback_cfs
else:
    raise FileNotFoundError(
        f"Could not find {preferred_cfs.name} or {fallback_cfs.name} in {CFS_DIR}."
    )

CFS_FILENAME = CFS_PATH.name
CFS_META_PATH = CFS_DIR / CFS_META_FILENAME
MODEL_PATH = ART_DIR / SURROGATE_MODEL_FILENAME
SURROGATE_META_PATH = ART_DIR / SURROGATE_META_FILENAME

FIG_DIR = ROOT / "figures_cfs_evaluation"
TABLE_DIR = ROOT / "tables_cfs_evaluation"
EVAL_DIR = ART_DIR / "mvtec_counterfactuals_evaluation"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("\nResolved paths:")
print("ROOT:", ROOT)
print("ART_DIR:", ART_DIR)
print("CFS_DIR:", CFS_DIR)
print("SPLITS_DIR:", SPLITS_DIR)
print("Selected CF file:", CFS_FILENAME)
print("CFS_PATH:", CFS_PATH)
print("CFS_META_PATH:", CFS_META_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("SURROGATE_META_PATH:", SURROGATE_META_PATH)

print("\nFile checks:")
print("Selected CFs:", CFS_PATH.exists())
print("CF metadata:", CFS_META_PATH.exists())
print("Surrogate model:", MODEL_PATH.exists())
print("Surrogate metadata:", SURROGATE_META_PATH.exists())
print("Splits dir:", SPLITS_DIR.exists())

assert CFS_PATH.is_file(), CFS_PATH
assert MODEL_PATH.is_file(), MODEL_PATH
assert SPLITS_DIR.is_dir(), SPLITS_DIR

Device: cuda
Current working directory:
/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class

Candidate MVTec roots:
- ok=1     | /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class
- ok=0     | /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/hpo_mvtec_4class
- ok=0     | /home/jovyan/fast/x_peer_pdt_hyperoptxai

Resolved paths:
ROOT: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class
ART_DIR: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata
CFS_DIR: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals
SPLITS_DIR: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits
Selected CF file: mvtec_all_methods_cfs.csv
CFS_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals/mvtec_all_methods_cfs.csv
CFS_META_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvte

In [2]:
# ============================================================
# 1) Load metadata and generated counterfactuals
# ============================================================

if CFS_META_PATH.exists():
    with open(CFS_META_PATH, "r", encoding="utf-8") as f:
        cf_meta = json.load(f)
else:
    cf_meta = {}

if SURROGATE_META_PATH.exists():
    with open(SURROGATE_META_PATH, "r", encoding="utf-8") as f:
        surrogate_meta = json.load(f)
else:
    surrogate_meta = {}

FEATURES = cf_meta.get("features", surrogate_meta.get("features"))
CATEGORICAL = cf_meta.get("categorical", surrogate_meta.get("categorical_features"))
NUMERIC = cf_meta.get("numeric", surrogate_meta.get("numeric_features"))
TARGET = cf_meta.get("target", surrogate_meta.get("target", "value"))

if FEATURES is None or CATEGORICAL is None or NUMERIC is None:
    raise ValueError(
        "Could not load feature schema from metadata. "
        "Check mvtec_all_methods_cfs_metadata.json and mvtec_surrogate_metadata.json."
    )

cfs = pd.read_csv(CFS_PATH)

if "cf_id" not in cfs.columns:
    if "global_cf_id" in cfs.columns:
        cfs["cf_id"] = pd.to_numeric(cfs["global_cf_id"], errors="coerce").astype(int)
    else:
        cfs["cf_id"] = np.arange(len(cfs), dtype=int)

if "method" not in cfs.columns:
    filename_lower = CFS_FILENAME.lower()
    if "dice" in filename_lower:
        cfs["method"] = "DiCE"
    elif "cfnow" in filename_lower:
        cfs["method"] = "CFNOW"
    elif "acfx" in filename_lower:
        cfs["method"] = "ACFX"
    else:
        cfs["method"] = "Unknown"

missing = [c for c in ["cf_id", TARGET] + FEATURES if c not in cfs.columns]
if missing:
    raise ValueError(f"Missing columns in CF file {CFS_FILENAME}: {missing}")

for c in CATEGORICAL:
    cfs[c] = cfs[c].astype(str)

for c in NUMERIC + [TARGET]:
    cfs[c] = pd.to_numeric(cfs[c], errors="coerce")

if "cf_method_score" in cfs.columns:
    cfs["cf_method_score"] = pd.to_numeric(cfs["cf_method_score"], errors="coerce")
else:
    cfs["cf_method_score"] = np.nan
    print("Warning: cf_method_score is missing. Rerun 4b_cfs_calculation.ipynb with the fixed export cell.")

if "surrogate_pred" in cfs.columns:
    cfs["surrogate_pred_file"] = pd.to_numeric(cfs["surrogate_pred"], errors="coerce")
else:
    cfs["surrogate_pred_file"] = np.nan

cfs = cfs.dropna(subset=[TARGET] + FEATURES).reset_index(drop=True)

print("Counterfactuals loaded:", cfs.shape)
print("Source file:", CFS_FILENAME)
print("Target:", TARGET)
print("Features:", len(FEATURES))
print("Categorical:", CATEGORICAL)
print("Numeric:", NUMERIC)

print("\nMethod counts:")
display(cfs["method"].value_counts(dropna=False))

score_cols = [
    "cf_id",
    "method",
    "cf_method_score",
    "surrogate_pred_file",
    TARGET,
    "strictly_valid",
]
score_cols = [c for c in score_cols if c in cfs.columns]
display(cfs[score_cols].round(6))

Counterfactuals loaded: (24, 31)
Source file: mvtec_all_methods_cfs.csv
Target: value
Features: 13
Categorical: ['params_backbone', 'params_batch_size', 'params_center_crop_key', 'params_image_size_key', 'params_layers_key', 'params_max_patches_per_image', 'params_pre_trained', 'params_reduction', 'params_soft_corruption_level']
Numeric: ['params_coreset_sampling_ratio', 'params_num_neighbors', 'params_soft_review_budget', 'params_soft_train_fraction']

Method counts:


DiCE     8
CFNOW    8
ACFX     8
Name: method, dtype: int64

,cf_id,method,cf_method_score,surrogate_pred_file,value,strictly_valid
0,0,DiCE,0.794842,0.794842,0.794842,True
1,1,DiCE,0.782018,0.782018,0.782018,True
2,2,DiCE,0.777679,0.777679,0.777679,True
3,3,DiCE,0.773289,0.773289,0.773289,True
4,4,DiCE,0.768456,0.768456,0.768456,True
5,5,DiCE,0.766264,0.766264,0.766264,True
6,6,DiCE,0.762813,0.762813,0.762813,True
7,7,DiCE,0.760298,0.760298,0.760298,True
8,8,CFNOW,0.819979,0.822756,0.822756,True
9,9,CFNOW,0.814026,0.819806,0.819806,True


In [3]:
# ============================================================
# 2) Surrogate consistency check
# ============================================================

cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

X_cf = cfs[FEATURES].copy()

for c in CATEGORICAL:
    X_cf[c] = X_cf[c].astype(str)

for c in NUMERIC:
    X_cf[c] = pd.to_numeric(X_cf[c], errors="coerce")

cfs["surrogate_pred_fresh"] = np.asarray(cb.predict(X_cf), dtype=float)

# Keep this name for compatibility with later evaluation/plotting cells.
cfs["surrogate_pred"] = cfs["surrogate_pred_fresh"]

cfs["gap_fresh_surrogate_minus_file_surrogate"] = (
    cfs["surrogate_pred_fresh"] - cfs["surrogate_pred_file"]
)

cfs["gap_fresh_surrogate_minus_method_score"] = (
    cfs["surrogate_pred_fresh"] - cfs["cf_method_score"]
)

cfs["gap_fresh_surrogate_minus_target_value"] = (
    cfs["surrogate_pred_fresh"] - pd.to_numeric(cfs[TARGET], errors="coerce")
)

display_cols = [
    "cf_id",
    "method",
    "cf_method_score",
    "surrogate_pred_file",
    "surrogate_pred_fresh",
    TARGET,
    "gap_fresh_surrogate_minus_method_score",
    "gap_fresh_surrogate_minus_file_surrogate",
    "gap_fresh_surrogate_minus_target_value",
    "strictly_valid",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
]

display_cols = [c for c in display_cols if c in cfs.columns]
display(cfs[display_cols].round(6))

print("\nFresh surrogate minus CF-method score:")
display(cfs["gap_fresh_surrogate_minus_method_score"].describe())

print("\nFresh surrogate minus stored surrogate:")
display(cfs["gap_fresh_surrogate_minus_file_surrogate"].describe())

print("\nFresh surrogate minus TARGET/value:")
display(cfs["gap_fresh_surrogate_minus_target_value"].describe())

,cf_id,method,cf_method_score,surrogate_pred_file,surrogate_pred_fresh,value,gap_fresh_surrogate_minus_method_score,gap_fresh_surrogate_minus_file_surrogate,gap_fresh_surrogate_minus_target_value,strictly_valid,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level
0,0,DiCE,0.794842,0.794842,0.794842,0.794842,-0.000000,0.0,0.0,True,0.499,0.70,mild
1,1,DiCE,0.782018,0.782018,0.782018,0.782018,-0.000000,0.0,0.0,True,0.494,0.75,mild
2,2,DiCE,0.777679,0.777679,0.777679,0.777679,0.000000,0.0,0.0,True,0.496,0.75,mild
3,3,DiCE,0.773289,0.773289,0.773289,0.773289,-0.000000,0.0,0.0,True,0.489,0.75,mild
4,4,DiCE,0.768456,0.768456,0.768456,0.768456,0.000000,0.0,0.0,True,0.494,0.75,mild
5,5,DiCE,0.766264,0.766264,0.766264,0.766264,-0.000000,0.0,0.0,True,0.485,0.52,mild
6,6,DiCE,0.762813,0.762813,0.762813,0.762813,0.000000,0.0,0.0,True,0.486,0.75,none
7,7,DiCE,0.760298,0.760298,0.760298,0.760298,0.000000,0.0,0.0,True,0.478,0.67,mild
8,8,CFNOW,0.819979,0.822756,0.822756,0.822756,0.002777,0.0,0.0,True,0.500,0.77,none
9,9,CFNOW,0.814026,0.819806,0.819806,0.819806,0.005780,0.0,0.0,True,0.500,0.76,none



Fresh surrogate minus CF-method score:


count    1.600000e+01
mean     2.123049e-05
std      3.187483e-03
min     -5.997716e-03
25%     -1.100589e-03
50%     -7.503626e-09
75%      2.445488e-08
max      6.894846e-03
Name: gap_fresh_surrogate_minus_method_score, dtype: float64


Fresh surrogate minus stored surrogate:


count    24.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: gap_fresh_surrogate_minus_file_surrogate, dtype: float64


Fresh surrogate minus TARGET/value:


count    24.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: gap_fresh_surrogate_minus_target_value, dtype: float64

In [4]:
# ============================================================
# 3) Load MVTec per-class train/validation/test splits
# ============================================================

CLASSES = ["bottle", "cable", "capsule", "hazelnut"]

print("SPLITS_DIR:", SPLITS_DIR)
print("Files/folders in SPLITS_DIR:")
for p in sorted(SPLITS_DIR.iterdir()):
    print("-", p.name)


def resolve_path_with_context(path_value, csv_parent_dir: Path):
    """
    Resolve image paths stored in the split CSVs.
    Handles absolute paths and common relative-path cases.
    """
    p = Path(str(path_value))

    if p.exists():
        return str(p.resolve())

    candidates = [
        csv_parent_dir / p,
        SPLITS_DIR / p,
        ROOT / p,
        ROOT.parent / p,
        ROOT / "mvtec_splits" / p,
        ROOT.parent / "hpo_mvtec_4class" / p,
    ]

    for c in candidates:
        if c.exists():
            return str(c.resolve())

    # Keep original for later diagnostics if not found.
    return str(path_value)


def load_class_split(class_name: str, split_filename: str, split_name: str):
    class_dir = SPLITS_DIR / class_name
    csv_path = class_dir / split_filename

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")

    d = pd.read_csv(csv_path)

    if "path" not in d.columns:
        raise ValueError(f"{csv_path} is missing required column: path")

    # Add class name if missing.
    if "class_name" not in d.columns:
        d["class_name"] = class_name

    # Add split name.
    d["split"] = split_name

    # Train-good split should be normal only.
    if "label" not in d.columns:
        if split_name == "train_good":
            d["label"] = 0
        else:
            raise ValueError(f"{csv_path} is missing required column: label")

    d["label"] = pd.to_numeric(d["label"], errors="coerce").astype(int)

    # Resolve paths relative to the class split folder if needed.
    d["path"] = d["path"].apply(lambda x: resolve_path_with_context(x, class_dir))

    d["source_split_file"] = str(csv_path)

    return d


train_parts = []
val_parts = []
final_parts = []

for cls in CLASSES:
    class_dir = SPLITS_DIR / cls

    if not class_dir.exists():
        raise FileNotFoundError(f"Missing class split folder: {class_dir}")

    train_parts.append(load_class_split(cls, "df_train_good.csv", "train_good"))
    val_parts.append(load_class_split(cls, "df_val.csv", "val"))
    final_parts.append(load_class_split(cls, "df_final.csv", "final_test"))

df_train_good = pd.concat(train_parts, ignore_index=True)
df_val = pd.concat(val_parts, ignore_index=True)
df_final = pd.concat(final_parts, ignore_index=True)

# Save combined split files for reproducibility and easier future loading.
COMBINED_SPLIT_DIR = SPLITS_DIR / "_combined"
COMBINED_SPLIT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = COMBINED_SPLIT_DIR / "df_train_good.csv"
VAL_PATH = COMBINED_SPLIT_DIR / "df_val.csv"
TEST_PATH = COMBINED_SPLIT_DIR / "df_final.csv"

df_train_good.to_csv(TRAIN_PATH, index=False)
df_val.to_csv(VAL_PATH, index=False)
df_final.to_csv(TEST_PATH, index=False)

print("\nCombined split files saved:")
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:", VAL_PATH)
print("TEST_PATH:", TEST_PATH)

print("\nSplit shapes:")
print("train_good:", df_train_good.shape)
print("val:", df_val.shape)
print("test:", df_final.shape)

display(df_train_good.head())
display(df_val.head())
display(df_final.head())

print("\nLabel distributions:")
print("train_good:")
display(df_train_good["label"].value_counts().sort_index())

print("val:")
display(df_val["label"].value_counts().sort_index())

print("test:")
display(df_final["label"].value_counts().sort_index())

print("\nClass/label distribution: validation")
display(pd.crosstab(df_val["class_name"], df_val["label"]))

print("\nClass/label distribution: final test")
display(pd.crosstab(df_final["class_name"], df_final["label"]))

print("\nPath existence rates:")
print("train_good:", np.mean([Path(p).exists() for p in df_train_good["path"]]))
print("val:", np.mean([Path(p).exists() for p in df_val["path"]]))
print("test:", np.mean([Path(p).exists() for p in df_final["path"]]))

SPLITS_DIR: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits
Files/folders in SPLITS_DIR:
- _combined
- bottle
- cable
- capsule
- hazelnut
- split_meta.json

Combined split files saved:
TRAIN_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/_combined/df_train_good.csv
VAL_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/_combined/df_val.csv
TEST_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/_combined/df_final.csv

Split shapes:
train_good: (1043, 7)
val: (237, 7)
test: (238, 7)


,path,label,mvtec_class,split,defect_type,class_name,source_split_file
0,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/train/good/000.png,0,bottle,train_good,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_train_good.csv
1,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/train/good/001.png,0,bottle,train_good,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_train_good.csv
2,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/train/good/002.png,0,bottle,train_good,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_train_good.csv
3,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/train/good/003.png,0,bottle,train_good,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_train_good.csv
4,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/train/good/004.png,0,bottle,train_good,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_train_good.csv


,path,label,mvtec_class,split,defect_type,class_name,source_split_file
0,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_large/000.png,1,bottle,val,broken_large,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_val.csv
1,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/good/005.png,0,bottle,val,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_val.csv
2,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_small/020.png,1,bottle,val,broken_small,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_val.csv
3,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/contamination/007.png,1,bottle,val,contamination,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_val.csv
4,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/contamination/000.png,1,bottle,val,contamination,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_val.csv


,path,label,mvtec_class,split,defect_type,class_name,source_split_file
0,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/good/010.png,0,bottle,final_test,good,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_final.csv
1,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_small/021.png,1,bottle,final_test,broken_small,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_final.csv
2,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_small/001.png,1,bottle,final_test,broken_small,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_final.csv
3,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_small/007.png,1,bottle,final_test,broken_small,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_final.csv
4,/home/jovyan/shared/xautoml_user/shared/mvtec_anomaly_detection/bottle/test/broken_small/016.png,1,bottle,final_test,broken_small,bottle,/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/mvtec_splits/bottle/df_final.csv



Label distributions:
train_good:


...


Class/label distribution: validation


label,0,1
class_name,,
bottle,10,31
cable,29,46
capsule,11,55
hazelnut,20,35



Class/label distribution: final test


label,0,1
class_name,,
bottle,10,32
cable,29,46
capsule,12,54
hazelnut,20,35



Path existence rates:
train_good: 1.0
val: 1.0
test: 1.0


In [5]:
# ============================================================
# 4) Validate combined split schema and image paths
# ============================================================

for name, d in [
    ("train_good", df_train_good),
    ("val", df_val),
    ("test", df_final),
]:
    required_cols = ["path", "label", "class_name"]

    missing = [c for c in required_cols if c not in d.columns]
    if missing:
        raise ValueError(f"{name} split is missing required columns: {missing}")

    d["label"] = pd.to_numeric(d["label"], errors="coerce").astype(int)

    exists_mask = d["path"].apply(lambda p: Path(p).exists())
    exists_rate = exists_mask.mean()

    print(f"{name}: shape={d.shape}, path_exists_rate={exists_rate:.3f}")

    if exists_rate < 1.0:
        print(f"\nMissing paths in {name}:")
        display(d.loc[~exists_mask, ["path", "class_name", "label"]].head(20))
        raise FileNotFoundError(f"Some image paths do not exist in {name} split.")

print("\nFinal split validation passed.")

print("\nTrain-good class distribution:")
display(df_train_good["class_name"].value_counts().sort_index())

print("\nValidation class/label distribution:")
display(pd.crosstab(df_val["class_name"], df_val["label"]))

print("\nFinal-test class/label distribution:")
display(pd.crosstab(df_final["class_name"], df_final["label"]))

train_good: shape=(1043, 7), path_exists_rate=1.000
val: shape=(237, 7), path_exists_rate=1.000
test: shape=(238, 7), path_exists_rate=1.000

Final split validation passed.

Train-good class distribution:


bottle      209
cable       224
capsule     219
hazelnut    391
Name: class_name, dtype: int64


Validation class/label distribution:


label,0,1
class_name,,
bottle,10,31
cable,29,46
capsule,11,55
hazelnut,20,35



Final-test class/label distribution:


label,0,1
class_name,,
bottle,10,32
cable,29,46
capsule,12,54
hazelnut,20,35


In [6]:
# ============================================================
# 5) PatchCore utilities
# ============================================================

def _stable_int_seed(text: str, seed: int) -> int:
    return (zlib.crc32(text.encode("utf-8")) + seed) & 0xFFFFFFFF


def corrupt_pil(img: Image.Image, level: str, seed_int: int) -> Image.Image:
    if level == "none":
        return img

    rng = np.random.default_rng(seed_int)
    out = img

    if level in {"mild", "strong"}:
        radius = 0.6 if level == "mild" else 1.2
        out = out.filter(ImageFilter.GaussianBlur(radius=radius))

    if level == "strong":
        arr = np.asarray(out).astype(np.float32)
        noise_std = 8.0
        noise = rng.normal(0.0, noise_std, size=arr.shape).astype(np.float32)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out


class ImagePathDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        preproc: transforms.Compose,
        corruption_level: str = "none",
        apply_corruption: bool = False,
    ):
        self.paths = df["path"].tolist()
        self.labels = df["label"].to_numpy().astype(int)
        self.preproc = preproc
        self.corruption_level = corruption_level
        self.apply_corruption = apply_corruption

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.apply_corruption and self.corruption_level != "none":
            s = _stable_int_seed(path, SEED)
            img = corrupt_pil(img, self.corruption_level, s)

        x = self.preproc(img)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_preproc(image_size: tuple[int, int], center_crop_size: tuple[int, int] | None):
    t = [transforms.Resize(image_size)]

    if center_crop_size is not None:
        t.append(transforms.CenterCrop(center_crop_size))

    t += [
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]

    return transforms.Compose(t)


def make_loaders(
    df_train_good_sub: pd.DataFrame,
    image_size: tuple[int, int],
    center_crop_size: tuple[int, int] | None,
    batch_size: int,
    corruption_level: str,
):
    preproc = make_preproc(
        image_size=image_size,
        center_crop_size=center_crop_size,
    )

    train_ds = ImagePathDataset(
        df_train_good_sub,
        preproc,
        corruption_level=corruption_level,
        apply_corruption=True,
    )

    val_ds = ImagePathDataset(
        df_val,
        preproc,
        corruption_level="none",
        apply_corruption=False,
    )

    test_ds = ImagePathDataset(
        df_final,
        preproc,
        corruption_level="none",
        apply_corruption=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, test_loader

In [7]:
# ============================================================
# 6) PatchCore feature extraction and scoring
# ============================================================

def build_backbone(backbone: str, pre_trained: bool) -> torch.nn.Module:
    if backbone == "resnet18":
        m = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT if pre_trained else None
        )
    elif backbone == "resnet34":
        m = models.resnet34(
            weights=models.ResNet34_Weights.DEFAULT if pre_trained else None
        )
    elif backbone == "resnet50":
        m = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pre_trained else None
        )
    elif backbone == "wide_resnet50_2":
        m = models.wide_resnet50_2(
            weights=models.Wide_ResNet50_2_Weights.DEFAULT if pre_trained else None
        )
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    m.eval()

    for p in m.parameters():
        p.requires_grad = False

    return m


class FeatureExtractor(torch.nn.Module):
    def __init__(self, backbone: str, layers: tuple[str, ...], pre_trained: bool):
        super().__init__()

        self.backbone = build_backbone(backbone, pre_trained)
        self.layers = layers
        self._features: dict[str, torch.Tensor] = {}

        named = dict(self.backbone.named_modules())

        for name in layers:
            if name not in named:
                raise ValueError(f"Layer '{name}' not found in backbone '{backbone}'")
            named[name].register_forward_hook(self._hook(name))

    def _hook(self, name: str):
        def fn(module, inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        self._features = {}
        _ = self.backbone(x)
        return [self._features[name] for name in self.layers]


def to_patch_embeddings(
    feat_maps: list[torch.Tensor],
    max_patches_per_image: int | None = None,
    seed: int = 0,
) -> torch.Tensor:
    H = max(f.shape[-2] for f in feat_maps)
    W = max(f.shape[-1] for f in feat_maps)

    resized = []

    for f in feat_maps:
        if (f.shape[-2], f.shape[-1]) != (H, W):
            f = torch.nn.functional.interpolate(
                f,
                size=(H, W),
                mode="bilinear",
                align_corners=False,
            )
        resized.append(f)

    f_cat = torch.cat(resized, dim=1)
    B, D, _, _ = f_cat.shape

    f_flat = f_cat.flatten(2).transpose(1, 2).contiguous()
    P = f_flat.shape[1]

    if max_patches_per_image is not None and max_patches_per_image < P:
        g = torch.Generator(device=f_flat.device)
        g.manual_seed(seed)
        idx = torch.randperm(P, generator=g, device=f_flat.device)[:max_patches_per_image]
        f_flat = f_flat[:, idx, :]

    return f_flat.reshape(-1, D)


def coreset_sample(embeddings: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    n = embeddings.shape[0]
    m = max(1, int(n * ratio))

    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=m, replace=False)

    return embeddings[idx]


@torch.no_grad()
def fit_memory_bank(
    extractor: FeatureExtractor,
    train_loader: DataLoader,
    coreset_sampling_ratio: float,
    max_patches_per_image: int,
) -> np.ndarray:
    extractor = extractor.to(DEVICE).eval()

    kept = []

    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(
            feats,
            max_patches_per_image=max_patches_per_image,
            seed=SEED,
        )

        kept.append(patches.detach().cpu().numpy())

        del x, feats, patches

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    emb = np.vstack(kept)

    return coreset_sample(
        emb,
        ratio=coreset_sampling_ratio,
        seed=SEED,
    )


@torch.no_grad()
def score_images(
    extractor: FeatureExtractor,
    loader: DataLoader,
    memory_bank: np.ndarray,
    num_neighbors: int,
    reduction: str,
    max_patches_per_image: int,
) -> tuple[np.ndarray, np.ndarray]:
    extractor = extractor.to(DEVICE).eval()

    nn = NearestNeighbors(
        n_neighbors=num_neighbors,
        algorithm="auto",
    )

    nn.fit(memory_bank)

    y_true_list = []
    y_score_list = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = (
            to_patch_embeddings(
                feats,
                max_patches_per_image=max_patches_per_image,
                seed=SEED,
            )
            .detach()
            .cpu()
            .numpy()
        )

        dists, _ = nn.kneighbors(patches, return_distance=True)
        patch_scores = dists.mean(axis=1)

        B = y.shape[0]
        P = patch_scores.shape[0] // B
        patch_scores = patch_scores.reshape(B, P)

        if reduction == "max":
            img_scores = patch_scores.max(axis=1)
        elif reduction == "mean":
            img_scores = patch_scores.mean(axis=1)
        else:
            raise ValueError(f"Unsupported reduction: {reduction}")

        y_true_list.append(y.numpy())
        y_score_list.append(img_scores)

        del x, feats, patches

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(y_true_list), np.concatenate(y_score_list)


def threshold_from_review_budget(y_score: np.ndarray, budget: float) -> float:
    assert 0.0 < budget < 1.0

    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))

    thr = np.partition(y_score, -k)[-k]

    return float(thr)

In [8]:
# ============================================================
# 7) Configuration decoding + real PatchCore evaluator
# ============================================================

LAYERS_MAP = {
    "l2_l3": ("layer2", "layer3"),
    "l1_l2_l3": ("layer1", "layer2", "layer3"),
    "l3": ("layer3",),
    "l2": ("layer2",),
}

IMAGE_SIZE_MAP = {
    "224": (224, 224),
    "256": (256, 256),
    "320": (320, 320),
}

def _as_bool(x):
    if isinstance(x, bool):
        return x

    s = str(x).strip().lower()

    return s in {"1", "true", "yes", "y", "t"}


def decode_patchcore_config(cfg: dict) -> dict:
    backbone = str(cfg["params_backbone"])
    batch_size = int(float(cfg["params_batch_size"]))
    center_crop_key = str(cfg["params_center_crop_key"])
    coreset_sampling_ratio = float(cfg["params_coreset_sampling_ratio"])
    image_size_key = str(cfg["params_image_size_key"])
    layers_key = str(cfg["params_layers_key"])
    max_patches_per_image = int(float(cfg["params_max_patches_per_image"]))
    num_neighbors = int(float(cfg["params_num_neighbors"]))
    pre_trained = _as_bool(cfg["params_pre_trained"])
    reduction = str(cfg["params_reduction"])

    soft_corruption_level = str(cfg["params_soft_corruption_level"])
    soft_review_budget = float(cfg["params_soft_review_budget"])
    soft_train_fraction = float(cfg["params_soft_train_fraction"])

    if image_size_key not in IMAGE_SIZE_MAP:
        raise ValueError(f"Unsupported image_size_key: {image_size_key}")

    if layers_key not in LAYERS_MAP:
        raise ValueError(f"Unsupported layers_key: {layers_key}")

    image_size = IMAGE_SIZE_MAP[image_size_key]
    layers = LAYERS_MAP[layers_key]

    if center_crop_key == "none":
        center_crop_size = None
    elif center_crop_key == "0.875":
        s = image_size[0]
        center_crop_size = (int(0.875 * s), int(0.875 * s))
    else:
        raise ValueError(f"Unsupported center_crop_key: {center_crop_key}")

    return {
        "backbone": backbone,
        "batch_size": batch_size,
        "center_crop_key": center_crop_key,
        "center_crop_size": center_crop_size,
        "coreset_sampling_ratio": coreset_sampling_ratio,
        "image_size_key": image_size_key,
        "image_size": image_size,
        "layers_key": layers_key,
        "layers": layers,
        "max_patches_per_image": max_patches_per_image,
        "num_neighbors": num_neighbors,
        "pre_trained": pre_trained,
        "reduction": reduction,
        "soft_corruption_level": soft_corruption_level,
        "soft_review_budget": soft_review_budget,
        "soft_train_fraction": soft_train_fraction,
    }


def run_patchcore_and_score(cfg: dict, seed_offset: int = 0) -> dict:
    decoded = decode_patchcore_config(cfg)

    soft_train_fraction = decoded["soft_train_fraction"]

    if soft_train_fraction >= 1.0:
        df_sub = df_train_good.copy()
    else:
        df_sub = df_train_good.sample(
            frac=soft_train_fraction,
            random_state=SEED + seed_offset,
        ).reset_index(drop=True)

    train_loader, val_loader, test_loader = make_loaders(
        df_train_good_sub=df_sub,
        image_size=decoded["image_size"],
        center_crop_size=decoded["center_crop_size"],
        batch_size=decoded["batch_size"],
        corruption_level=decoded["soft_corruption_level"],
    )

    t0 = time.time()

    extractor = FeatureExtractor(
        backbone=decoded["backbone"],
        layers=decoded["layers"],
        pre_trained=decoded["pre_trained"],
    )

    memory_bank = fit_memory_bank(
        extractor,
        train_loader,
        coreset_sampling_ratio=decoded["coreset_sampling_ratio"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )

    # Validation scores: calibrate threshold using review budget
    y_true_val, y_score_val = score_images(
        extractor,
        val_loader,
        memory_bank,
        num_neighbors=decoded["num_neighbors"],
        reduction=decoded["reduction"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )

    thr = threshold_from_review_budget(
        y_score=y_score_val,
        budget=decoded["soft_review_budget"],
    )

    y_pred_val = (y_score_val >= thr).astype(int)

    p_val, r_val, f1_val, _ = precision_recall_fscore_support(
        y_true_val,
        y_pred_val,
        average="binary",
        zero_division=0,
    )

    try:
        auroc_val = roc_auc_score(y_true_val, y_score_val)
    except ValueError:
        auroc_val = np.nan

    # Test scores: apply validation-calibrated threshold
    y_true_test, y_score_test = score_images(
        extractor,
        test_loader,
        memory_bank,
        num_neighbors=decoded["num_neighbors"],
        reduction=decoded["reduction"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )

    y_pred_test = (y_score_test >= thr).astype(int)

    p_test, r_test, f1_test, _ = precision_recall_fscore_support(
        y_true_test,
        y_pred_test,
        average="binary",
        zero_division=0,
    )

    try:
        auroc_test = roc_auc_score(y_true_test, y_score_test)
    except ValueError:
        auroc_test = np.nan

    runtime_sec = time.time() - t0

    # Clean up GPU memory
    del extractor, memory_bank

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "threshold_val_budget": float(thr),

        "precision_val": float(p_val),
        "recall_val": float(r_val),
        "f1_val": float(f1_val),
        "auroc_val": float(auroc_val) if not pd.isna(auroc_val) else np.nan,

        "precision_test": float(p_test),
        "recall_test": float(r_test),
        "f1_test": float(f1_test),
        "auroc_test": float(auroc_test) if not pd.isna(auroc_test) else np.nan,

        "runtime_sec": float(runtime_sec),
        "train_good_used": int(len(df_sub)),
        "memory_bank_size": int(memory_bank.shape[0]) if "memory_bank" in locals() else np.nan,
    }

In [9]:
# ============================================================
# 8) Select CFs to evaluate
# ============================================================

# Set to None to evaluate all returned CFs.
# For a quick smoke test, set e.g. MAX_CFS_TO_EVALUATE = 3.
# For paper plots, keep None so the all-method file is evaluated completely.
MAX_CFS_TO_EVALUATE = None

cfs_eval = cfs.copy()

sort_cols = []
ascending = []

if "strictly_valid" in cfs_eval.columns:
    sort_cols.append("strictly_valid")
    ascending.append(False)

if "method" in cfs_eval.columns:
    sort_cols.append("method")
    ascending.append(True)

sort_cols.append(TARGET)
ascending.append(False)

cfs_eval = cfs_eval.sort_values(by=sort_cols, ascending=ascending).reset_index(drop=True)

if MAX_CFS_TO_EVALUATE is not None:
    cfs_eval = cfs_eval.head(MAX_CFS_TO_EVALUATE).copy()

print("Counterfactuals selected for real PatchCore evaluation:", cfs_eval.shape)

if "method" in cfs_eval.columns:
    print("\nSelected CFs by method:")
    display(cfs_eval["method"].value_counts(dropna=False))

display_cols = [
    "cf_id",
    "method",
    "method_rank",
    TARGET,
    "surrogate_pred",
    "strictly_valid",
    "params_backbone",
    "params_image_size_key",
    "params_layers_key",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
]

display_cols = [c for c in display_cols if c in cfs_eval.columns]
display(cfs_eval[display_cols])

Counterfactuals selected for real PatchCore evaluation: (24, 35)

Selected CFs by method:


ACFX     8
CFNOW    8
DiCE     8
Name: method, dtype: int64

,cf_id,method,method_rank,value,surrogate_pred,strictly_valid,params_backbone,params_image_size_key,params_layers_key,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level
0,16,ACFX,0,0.794809,0.794809,True,resnet18,224,l2,0.495,0.89,mild
1,17,ACFX,1,0.794066,0.794066,True,resnet18,224,l2,0.495,0.74,mild
2,18,ACFX,2,0.790701,0.790701,True,resnet18,224,l2,0.495,0.84,mild
3,19,ACFX,3,0.790506,0.790506,True,resnet18,224,l2,0.495,0.75,mild
4,20,ACFX,4,0.786811,0.786811,True,resnet18,224,l2,0.495,0.79,mild
5,21,ACFX,5,0.780857,0.780857,True,resnet18,224,l2,0.495,0.73,mild
6,22,ACFX,6,0.753426,0.753426,True,resnet18,224,l2,0.495,0.89,mild
7,23,ACFX,7,0.739636,0.739636,True,resnet18,224,l2,0.480,0.64,mild
8,8,CFNOW,0,0.822756,0.822756,True,wide_resnet50_2,256,l3,0.500,0.77,none
9,9,CFNOW,1,0.819806,0.819806,True,wide_resnet50_2,256,l3,0.500,0.76,none


In [10]:
# ============================================================
# Helper: fresh surrogate prediction for generated CF configs
# ============================================================

def prepare_cf_for_surrogate(cf_df, feature_cols, categorical_cols=None, numeric_cols=None):
    """
    Align generated CF configurations to the exact schema expected by the surrogate.
    """
    x = cf_df.copy()

    extra_cols = [c for c in x.columns if c not in feature_cols]
    if extra_cols:
        x = x.drop(columns=extra_cols, errors="ignore")

    x = x.reindex(columns=feature_cols)

    if categorical_cols is not None:
        for c in categorical_cols:
            if c in x.columns:
                x[c] = x[c].astype(str)

    if numeric_cols is not None:
        for c in numeric_cols:
            if c in x.columns:
                x[c] = pd.to_numeric(x[c], errors="coerce")

    return x


def predict_surrogate_f1(row_or_df):
    """
    Fresh CatBoost surrogate prediction for one row or a dataframe of CF configurations.
    """
    if isinstance(row_or_df, pd.Series):
        x = pd.DataFrame([row_or_df[FEATURES].to_dict()])
    else:
        x = row_or_df[FEATURES].copy()

    x = prepare_cf_for_surrogate(
        cf_df=x,
        feature_cols=FEATURES,
        categorical_cols=CATEGORICAL,
        numeric_cols=NUMERIC,
    )

    pred = cb.predict(x[FEATURES])
    pred = np.asarray(pred, dtype=float)

    if len(pred) == 1:
        return float(pred[0])

    return pred

In [11]:
# ============================================================
# 9) Run real PatchCore evaluation for selected CFs
# ============================================================

for _old_name in ["cf_value", "surrogate_pred"]:
    if _old_name in globals():
        del globals()[_old_name]

rows = []

meta_cols_to_keep = [
    "method",
    "method_rank",
    "global_cf_id",
    "strictly_valid",
    "n_violations",
    "distance_to_factual",
    "generation_sec",
]

for i, (_, r) in enumerate(cfs_eval.iterrows(), start=1):
    cf_id = int(r["cf_id"])
    row_meta = {c: r[c] for c in meta_cols_to_keep if c in r.index}

    print("\n" + "=" * 80)
    print(f"Evaluating CF {cf_id} ({i}/{len(cfs_eval)})")
    if "method" in r.index:
        print("Method:", r["method"])
    print("=" * 80)

    cfg = r[FEATURES].to_dict()

    try:
        out = run_patchcore_and_score(
            cfg,
            seed_offset=cf_id,
        )

        cf_method_score = (
            float(r["cf_method_score"])
            if "cf_method_score" in r.index and pd.notna(r["cf_method_score"])
            else np.nan
        )

        cf_table_value = float(r[TARGET])

        surrogate_pred_file = (
            float(r["surrogate_pred_file"])
            if "surrogate_pred_file" in r.index and pd.notna(r["surrogate_pred_file"])
            else np.nan
        )

        surrogate_pred_fresh = predict_surrogate_f1(r)

        real_val_f1 = float(out["f1_val"])
        real_test_f1 = float(out["f1_test"])

        row = {
            "cf_id": cf_id,
            **row_meta,

            "cf_method_score": cf_method_score,
            "cf_table_value": cf_table_value,
            "surrogate_pred_file": surrogate_pred_file,
            "surrogate_pred_fresh": surrogate_pred_fresh,

            # Compatibility names for plotting.
            "cf_value": cf_method_score,
            "surrogate_pred": surrogate_pred_fresh,

            "gap_file_surrogate_minus_method_score": surrogate_pred_file - cf_method_score,
            "gap_fresh_surrogate_minus_method_score": surrogate_pred_fresh - cf_method_score,
            "gap_fresh_surrogate_minus_table_value": surrogate_pred_fresh - cf_table_value,

            "f1_val_real": real_val_f1,
            "f1_test": real_test_f1,
            "auroc_val": float(out["auroc_val"]) if not pd.isna(out["auroc_val"]) else np.nan,
            "auroc_test": float(out["auroc_test"]) if not pd.isna(out["auroc_test"]) else np.nan,

            "gap_test_minus_method_score": real_test_f1 - cf_method_score,
            "gap_test_minus_surrogate_pred": real_test_f1 - surrogate_pred_fresh,
            "gap_test_minus_table_value": real_test_f1 - cf_table_value,

            # Backward-compatible name. Now this means real test F1 minus CF-method score.
            "gap_test_minus_cf_value": real_test_f1 - cf_method_score,

            "runtime_sec": float(out["runtime_sec"]),
            "status": "ok",
            "error": "",
        }

        for k, v in out.items():
            if k not in row and not isinstance(v, (np.ndarray, list, tuple, dict)):
                row[k] = v

        for f in FEATURES:
            row[f] = r[f]

        rows.append(row)

        print(
            f"CF {cf_id}: "
            f"cf_method_score={cf_method_score:.6f}, "
            f"cf_table_value={cf_table_value:.6f}, "
            f"file_surrogate_f1={surrogate_pred_file:.6f}, "
            f"fresh_surrogate_f1={surrogate_pred_fresh:.6f}, "
            f"real_val_f1={real_val_f1:.6f}, "
            f"real_test_f1={real_test_f1:.6f}, "
            f"AUROC={row['auroc_test']:.6f}, "
            f"runtime={row['runtime_sec']:.2f}s"
        )

    except Exception as e:
        print(f"FAILED CF {cf_id}: {repr(e)}")

        row = {
            "cf_id": cf_id,
            **row_meta,
            "cf_method_score": float(r["cf_method_score"]) if "cf_method_score" in r.index and pd.notna(r["cf_method_score"]) else np.nan,
            "cf_table_value": float(r[TARGET]) if TARGET in r.index else np.nan,
            "surrogate_pred_file": float(r["surrogate_pred_file"]) if "surrogate_pred_file" in r.index and pd.notna(r["surrogate_pred_file"]) else np.nan,
            "surrogate_pred_fresh": np.nan,
            "cf_value": float(r["cf_method_score"]) if "cf_method_score" in r.index and pd.notna(r["cf_method_score"]) else np.nan,
            "surrogate_pred": np.nan,
            "status": "failed",
            "error": repr(e),
        }

        for f in FEATURES:
            row[f] = r[f]

        rows.append(row)

eval_df = pd.DataFrame(rows)

if "f1_test" in eval_df.columns:
    eval_df = eval_df.sort_values(
        by=["status", "f1_test"],
        ascending=[False, False],
    ).reset_index(drop=True)

print("\nEvaluation results:")
display(eval_df)


Evaluating CF 16 (1/24)
Method: ACFX
CF 16: cf_method_score=nan, cf_table_value=0.794809, file_surrogate_f1=0.794809, fresh_surrogate_f1=0.794809, real_val_f1=0.582456, real_test_f1=0.621160, AUROC=0.479126, runtime=33.66s

Evaluating CF 17 (2/24)
Method: ACFX
CF 17: cf_method_score=nan, cf_table_value=0.794066, file_surrogate_f1=0.794066, fresh_surrogate_f1=0.794066, real_val_f1=0.589474, real_test_f1=0.619048, AUROC=0.476680, runtime=28.78s

Evaluating CF 18 (3/24)
Method: ACFX
CF 18: cf_method_score=nan, cf_table_value=0.790701, file_surrogate_f1=0.790701, fresh_surrogate_f1=0.790701, real_val_f1=0.589474, real_test_f1=0.621160, AUROC=0.478367, runtime=33.10s

Evaluating CF 19 (4/24)
Method: ACFX
CF 19: cf_method_score=nan, cf_table_value=0.790506, file_surrogate_f1=0.790506, fresh_surrogate_f1=0.790506, real_val_f1=0.589474, real_test_f1=0.621160, AUROC=0.478452, runtime=29.54s

Evaluating CF 20 (5/24)
Method: ACFX
CF 20: cf_method_score=nan, cf_table_value=0.786811, file_surrogat

,cf_id,method,method_rank,global_cf_id,strictly_valid,n_violations,distance_to_factual,generation_sec,cf_method_score,cf_table_value,surrogate_pred_file,surrogate_pred_fresh,cf_value,surrogate_pred,gap_file_surrogate_minus_method_score,gap_fresh_surrogate_minus_method_score,gap_fresh_surrogate_minus_table_value,f1_val_real,f1_test,auroc_val,auroc_test,gap_test_minus_method_score,gap_test_minus_surrogate_pred,gap_test_minus_table_value,gap_test_minus_cf_value,runtime_sec,status,error,threshold_val_budget,precision_val,recall_val,f1_val,precision_test,recall_test,train_good_used,memory_bank_size,params_backbone,params_batch_size,params_center_crop_key,params_image_size_key,params_layers_key,params_max_patches_per_image,params_pre_trained,params_reduction,params_soft_corruption_level,params_coreset_sampling_ratio,params_num_neighbors,params_soft_review_budget,params_soft_train_fraction
0,1,DiCE,1,1,True,0,1.452525,0.364300,0.782018,0.782018,0.782018,0.782018,0.782018,0.782018,-1.002629e-08,-1.002629e-08,0.0,0.624561,0.638889,0.504277,0.487138,-0.143129,-0.143129,-0.143129,-0.143129,30.038199,ok,,1.595084,0.754237,0.532934,0.624561,0.760331,0.550898,782,NaN,resnet18,16,0.875,224,l2,1024,True,mean,mild,0.029,9.0,0.494,0.75
1,3,DiCE,3,3,True,0,1.442424,0.364300,0.773289,0.773289,0.773289,0.773289,0.773289,0.773289,-1.679311e-08,-1.679311e-08,0.0,0.614841,0.638889,0.503593,0.490259,-0.134400,-0.134400,-0.134400,-0.134400,29.779973,ok,,1.591015,0.750000,0.520958,0.614841,0.760331,0.550898,782,NaN,resnet18,16,0.875,224,l2,1024,True,mean,mild,0.029,9.0,0.489,0.75
2,4,DiCE,4,4,True,0,0.452525,0.364300,0.768456,0.768456,0.768456,0.768456,0.768456,0.768456,1.041202e-08,1.041202e-08,0.0,0.624561,0.638889,0.502994,0.487897,-0.129567,-0.129567,-0.129567,-0.129567,29.843406,ok,,1.633626,0.754237,0.532934,0.624561,0.760331,0.550898,782,NaN,resnet18,16,0.875,224,l2,512,True,mean,mild,0.029,9.0,0.494,0.75
3,2,DiCE,2,2,True,0,1.456566,0.364300,0.777679,0.777679,0.777679,0.777679,0.777679,0.777679,1.863735e-08,1.863735e-08,0.0,0.624561,0.629371,0.500513,0.479126,-0.148309,-0.148309,-0.148309,-0.148309,29.909641,ok,,1.639918,0.754237,0.532934,0.624561,0.756303,0.538922,782,NaN,resnet18,16,0.875,256,l2,512,True,mean,mild,0.029,9.0,0.496,0.75
4,21,ACFX,5,21,True,0,0.602994,99.836937,NaN,0.780857,0.780857,0.780857,NaN,0.780857,NaN,NaN,0.0,0.610526,0.625000,0.497776,0.487476,NaN,-0.155857,-0.155857,NaN,28.750179,ok,,1.575739,0.737288,0.520958,0.610526,0.743802,0.538922,761,NaN,resnet18,16,0.875,224,l2,512,True,mean,mild,0.027,6.0,0.495,0.73
5,6,DiCE,6,6,True,0,2.436364,0.364300,0.762813,0.762813,0.762813,0.762813,0.762813,0.762813,2.909277e-08,2.909277e-08,0.0,0.614841,0.624561,0.499572,0.481150,-0.138252,-0.138252,-0.138252,-0.138252,18.170678,ok,,1.643360,0.750000,0.520958,0.614841,0.754237,0.532934,782,NaN,resnet18,16,0.875,256,l2,512,True,mean,none,0.029,9.0,0.486,0.75
6,16,ACFX,0,16,True,0,0.945408,99.836937,NaN,0.794809,0.794809,0.794809,NaN,0.794809,NaN,NaN,0.0,0.582456,0.621160,0.486741,0.479126,NaN,-0.173649,-0.173649,NaN,33.658529,ok,,1.358941,0.703390,0.497006,0.582456,0.722222,0.544910,928,NaN,resnet18,16,0.875,224,l2,512,True,mean,mild,0.033,1.0,0.495,0.89
7,18,ACFX,2,18,True,0,1.424976,99.836937,NaN,0.790701,0.790701,0.790701,NaN,0.790701,NaN,NaN,0.0,0.589474,0.621160,0.488537,0.478367,NaN,-0.169541,-0.169541,NaN,33.095788,ok,,1.378360,0.711864,0.502994,0.589474,0.722222,0.544910,876,NaN,resnet18,16,0.875,224,l2,512,True,mean,mild,0.101,5.0,0.495,0.84
8,19,ACFX,3,19,True,0,0.730408,99.836937,NaN,0.790506,0.790506,0.790506,NaN,0.790506,NaN,NaN,0.0,0.589474,0.621160,0.490590,0.478452,NaN,-0.169345,-0.169345,NaN,29.542557,ok,,1.388543,0.711864,0.502994,0.589474,0.722222,0.544910,782,NaN,resnet18,16,0.875,224,l2,512,True,mean,mild,0.029,1.0,0.495,0.75
9,17,ACFX,1,17,True,0,0.802908,99.836937,NaN,0.794066,0.794066,0.794066,NaN,0.794066,NaN,NaN,0.0,0.589474,0.619048,0.487682,0.476680,NaN,-0.175018,-0.175018,NaN,28.775279,ok,,1.370996,0.711864,0.

In [12]:
# ============================================================
# Diagnostic: check score semantics after real evaluation
# ============================================================

ok_df = eval_df[eval_df["status"] == "ok"].copy()

check_cols = [
    "cf_id",
    "method",
    "cf_method_score",
    "cf_table_value",
    "surrogate_pred_file",
    "surrogate_pred_fresh",
    "f1_val_real",
    "f1_test",
    "gap_fresh_surrogate_minus_method_score",
    "gap_test_minus_method_score",
    "gap_test_minus_surrogate_pred",
]

check_cols = [c for c in check_cols if c in ok_df.columns]

display(
    ok_df[check_cols]
    .sort_values(["method", "cf_id"])
    .round(6)
)

print("Fresh surrogate minus CF-method score:")
display(ok_df["gap_fresh_surrogate_minus_method_score"].describe())

print("Real test F1 minus fresh surrogate:")
display(ok_df["gap_test_minus_surrogate_pred"].describe())

print("Real test F1 minus CF-method score:")
display(ok_df["gap_test_minus_method_score"].describe())

,cf_id,method,cf_method_score,cf_table_value,surrogate_pred_file,surrogate_pred_fresh,f1_val_real,f1_test,gap_fresh_surrogate_minus_method_score,gap_test_minus_method_score,gap_test_minus_surrogate_pred
6,16,ACFX,NaN,0.794809,0.794809,0.794809,0.582456,0.621160,NaN,NaN,-0.173649
9,17,ACFX,NaN,0.794066,0.794066,0.794066,0.589474,0.619048,NaN,NaN,-0.175018
7,18,ACFX,NaN,0.790701,0.790701,0.790701,0.589474,0.621160,NaN,NaN,-0.169541
8,19,ACFX,NaN,0.790506,0.790506,0.790506,0.589474,0.621160,NaN,NaN,-0.169345
12,20,ACFX,NaN,0.786811,0.786811,0.786811,0.589474,0.602076,NaN,NaN,-0.184735
4,21,ACFX,NaN,0.780857,0.780857,0.780857,0.610526,0.625000,NaN,NaN,-0.155857
23,22,ACFX,NaN,0.753426,0.753426,0.753426,0.554386,0.564460,NaN,NaN,-0.188966
14,23,ACFX,NaN,0.739636,0.739636,0.739636,0.583630,0.579505,NaN,NaN,-0.160131
15,8,CFNOW,0.819979,0.822756,0.822756,0.822756,0.559441,0.575439,0.002777,-0.244540,-0.247317
21,9,CFNOW,0.814026,0.819806,0.819806,0.819806,0.559441,0.566434,0.005780,-0.247593,-0.253373


Fresh surrogate minus CF-method score:


count    1.600000e+01
mean     2.123049e-05
std      3.187483e-03
min     -5.997716e-03
25%     -1.100589e-03
50%     -7.503626e-09
75%      2.445488e-08
max      6.894846e-03
Name: gap_fresh_surrogate_minus_method_score, dtype: float64

Real test F1 minus fresh surrogate:


count    24.000000
mean     -0.189601
std       0.043744
min      -0.253373
25%      -0.241941
50%      -0.174333
75%      -0.157832
max      -0.129567
Name: gap_test_minus_surrogate_pred, dtype: float64

Real test F1 minus CF-method score:


count    16.000000
mean     -0.198302
std       0.051281
min      -0.252603
25%      -0.245876
50%      -0.211481
75%      -0.147014
max      -0.129567
Name: gap_test_minus_method_score, dtype: float64

In [13]:
# ============================================================
# 10) Summary tables
# ============================================================

summary_cols = [
    "cf_id",
    "method",
    "method_rank",
    "status",
    "cf_method_score",
    "cf_table_value",
    "surrogate_pred_file",
    "surrogate_pred_fresh",
    "surrogate_pred",
    "f1_val_real",
    "f1_test",
    "auroc_test",
    "gap_fresh_surrogate_minus_method_score",
    "gap_test_minus_method_score",
    "gap_test_minus_surrogate_pred",
    "runtime_sec",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
]

summary_cols = [c for c in summary_cols if c in eval_df.columns]

display(eval_df[summary_cols].round(6))

ok_df = eval_df[eval_df["status"] == "ok"].copy()

print("Successful evaluations:", len(ok_df), "/", len(eval_df))

if "method" in ok_df.columns and not ok_df.empty:
    print("\nSuccessful evaluations by method:")
    display(ok_df["method"].value_counts(dropna=False))

if not ok_df.empty:
    print("\nTest F1 summary:")
    display(ok_df["f1_test"].describe())

    print("\nAUROC summary:")
    display(ok_df["auroc_test"].describe())

    print("\nGap: fresh surrogate F1 - CF-method score")
    display(ok_df["gap_fresh_surrogate_minus_method_score"].describe())

    print("\nGap: real test F1 - CF-method score")
    display(ok_df["gap_test_minus_method_score"].describe())

    print("\nGap: real test F1 - fresh surrogate prediction")
    display(ok_df["gap_test_minus_surrogate_pred"].describe())

    print("\nRuntime summary:")
    display(ok_df["runtime_sec"].describe())

,cf_id,method,method_rank,status,cf_method_score,cf_table_value,surrogate_pred_file,surrogate_pred_fresh,surrogate_pred,f1_val_real,f1_test,auroc_test,gap_fresh_surrogate_minus_method_score,gap_test_minus_method_score,gap_test_minus_surrogate_pred,runtime_sec,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level
0,1,DiCE,1,ok,0.782018,0.782018,0.782018,0.782018,0.782018,0.624561,0.638889,0.487138,-0.000000,-0.143129,-0.143129,30.038199,0.494,0.75,mild
1,3,DiCE,3,ok,0.773289,0.773289,0.773289,0.773289,0.773289,0.614841,0.638889,0.490259,-0.000000,-0.134400,-0.134400,29.779973,0.489,0.75,mild
2,4,DiCE,4,ok,0.768456,0.768456,0.768456,0.768456,0.768456,0.624561,0.638889,0.487897,0.000000,-0.129567,-0.129567,29.843406,0.494,0.75,mild
3,2,DiCE,2,ok,0.777679,0.777679,0.777679,0.777679,0.777679,0.624561,0.629371,0.479126,0.000000,-0.148309,-0.148309,29.909641,0.496,0.75,mild
4,21,ACFX,5,ok,NaN,0.780857,0.780857,0.780857,0.780857,0.610526,0.625000,0.487476,NaN,NaN,-0.155857,28.750179,0.495,0.73,mild
5,6,DiCE,6,ok,0.762813,0.762813,0.762813,0.762813,0.762813,0.614841,0.624561,0.481150,0.000000,-0.138252,-0.138252,18.170678,0.486,0.75,none
6,16,ACFX,0,ok,NaN,0.794809,0.794809,0.794809,0.794809,0.582456,0.621160,0.479126,NaN,NaN,-0.173649,33.658529,0.495,0.89,mild
7,18,ACFX,2,ok,NaN,0.790701,0.790701,0.790701,0.790701,0.589474,0.621160,0.478367,NaN,NaN,-0.169541,33.095788,0.495,0.84,mild
8,19,ACFX,3,ok,NaN,0.790506,0.790506,0.790506,0.790506,0.589474,0.621160,0.478452,NaN,NaN,-0.169345,29.542557,0.495,0.75,mild
9,17,ACFX,1,ok,NaN,0.794066,0.794066,0.794066,0.794066,0.589474,0.619048,0.476680,NaN,NaN,-0.175018,28.775279,0.495,0.74,mild


Successful evaluations: 24 / 24

Successful evaluations by method:


DiCE     8
ACFX     8
CFNOW    8
Name: method, dtype: int64


Test F1 summary:


count    24.000000
mean      0.599413
std       0.028023
min       0.564460
25%       0.568421
50%       0.604925
75%       0.622011
max       0.638889
Name: f1_test, dtype: float64

...

count    16.000000
mean     -0.198302
std       0.051281
min      -0.252603
25%      -0.245876
50%      -0.211481
75%      -0.147014
max      -0.129567
Name: gap_test_minus_method_score, dtype: float64


Gap: real test F1 - fresh surrogate prediction


count    24.000000
mean     -0.189601
std       0.043744
min      -0.253373
25%      -0.241941
50%      -0.174333
75%      -0.157832
max      -0.129567
Name: gap_test_minus_surrogate_pred, dtype: float64


Runtime summary:


count    24.000000
mean     30.755870
std      11.020163
min      18.170678
25%      25.385548
50%      29.811690
75%      32.695126
max      60.132467
Name: runtime_sec, dtype: float64

In [37]:
# ============================================================
# 11) Paper-ready tables and plots
# ============================================================

if ok_df.empty:
    raise ValueError("No successful evaluations available for paper plots.")

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

saved_figures = []


def save_paper_fig(fig, stem):
    paths = []
    for ext in ["pdf", "svg", "png"]:
        out = FIG_DIR / f"{stem}.{ext}"
        fig.savefig(out, bbox_inches="tight")
        paths.append(out)
    plt.close(fig)
    saved_figures.extend(paths)
    return paths


paper_df = ok_df.copy()

if "method" not in paper_df.columns:
    paper_df["method"] = "DiCE" if "dice" in CFS_FILENAME.lower() else "Unknown"

paper_df["method"] = paper_df["method"].fillna("Unknown").astype(str)
paper_df["cf_id"] = paper_df["cf_id"].astype(int)
paper_df["candidate_label"] = np.where(
    paper_df["method"].eq("Unknown"),
    "CF " + paper_df["cf_id"].astype(str),
    paper_df["method"] + "-" + paper_df["cf_id"].astype(str),
)

paper_df = paper_df.sort_values(
    by=["method", "f1_test"],
    ascending=[True, False],
).reset_index(drop=True)

# -------- Compact paper table --------

paper_table_cols = [
    "candidate_label",
    "method",
    "cf_id",
    "cf_method_score",
    "surrogate_pred",
    "f1_val_real",
    "f1_test",
    "auroc_test",
    "gap_fresh_surrogate_minus_method_score",
    "gap_test_minus_method_score",
    "gap_test_minus_surrogate_pred",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
    "params_backbone",
    "params_image_size_key",
    "params_layers_key",
    "runtime_sec",
]

paper_table_cols = [c for c in paper_table_cols if c in paper_df.columns]
paper_table = paper_df[paper_table_cols].copy()

rename_map = {
    "candidate_label": "Candidate",
    "method": "Method",
    "cf_id": "CF id",
    "cf_method_score": "CF method score",
    "surrogate_pred": "Surrogate F1",
    "f1_val_real": "Real val F1",
    "f1_test": "Confirmed PatchCore F1",
    "auroc_test": "Test AUROC",
    "gap_fresh_surrogate_minus_method_score": "Surrogate - method",
    "gap_test_minus_method_score": "Test - method",
    "gap_test_minus_surrogate_pred": "Test - surrogate",
    "params_soft_review_budget": "Review budget",
    "params_soft_train_fraction": "Training fraction",
    "params_soft_corruption_level": "Corruption",
    "params_backbone": "Backbone",
    "params_image_size_key": "Image size",
    "params_layers_key": "Layers",
    "runtime_sec": "Runtime (s)",
}

paper_table = paper_table.rename(columns=rename_map)

round_cols = paper_table.select_dtypes(include=[np.number]).columns
paper_table[round_cols] = paper_table[round_cols].round(3)

paper_table_csv = TABLE_DIR / "mvtec_cf_confirmation_paper_table.csv"
paper_table_latex = TABLE_DIR / "mvtec_cf_confirmation_paper_table.tex"

paper_table.to_csv(paper_table_csv, index=False)
paper_table.to_latex(paper_table_latex, index=False, escape=True, float_format="%.3f")

print("Paper table:")
display(paper_table)
print("Saved paper table CSV:", paper_table_csv)
print("Saved paper table LaTeX:", paper_table_latex)

# -------- Plot 1: average method score / surrogate / confirmed PatchCore F1 by method --------

method_order = ["ACFX", "CFNOW", "DiCE"]
available_methods = [m for m in method_order if m in paper_df["method"].unique()]

if not available_methods:
    available_methods = sorted(paper_df["method"].unique())

method_summary = (
    paper_df
    .groupby("method")
    .agg(
        n=("cf_id", "count"),

        cf_method_score_mean=("cf_method_score", "mean"),
        cf_method_score_std=("cf_method_score", "std"),

        surrogate_pred_mean=("surrogate_pred", "mean"),
        surrogate_pred_std=("surrogate_pred", "std"),

        f1_test_mean=("f1_test", "mean"),
        f1_test_std=("f1_test", "std"),

        f1_val_real_mean=("f1_val_real", "mean"),
        f1_val_real_std=("f1_val_real", "std"),
    )
    .reindex(available_methods)
    .reset_index()
)

# For ACFX only: the method does not return a separate F1 score.
# For presentation, show the selected ACFX surrogate-scored candidate as the method score
# and hide the separate surrogate bar to avoid showing the same value twice.
acfx_mask = method_summary["method"].eq("ACFX")

method_summary.loc[acfx_mask, "cf_method_score_mean"] = method_summary.loc[
    acfx_mask, "surrogate_pred_mean"
]
method_summary.loc[acfx_mask, "cf_method_score_std"] = method_summary.loc[
    acfx_mask, "surrogate_pred_std"
]

method_summary.loc[acfx_mask, "surrogate_pred_mean"] = np.nan
method_summary.loc[acfx_mask, "surrogate_pred_std"] = np.nan

# Use the original surrogate values for the gap annotation.
gap_source = (
    paper_df
    .groupby("method")
    .agg(
        surrogate_for_gap=("surrogate_pred", "mean"),
        patchcore_for_gap=("f1_test", "mean"),
    )
    .reindex(available_methods)
    .reset_index()
)

method_summary["test_minus_surrogate"] = (
    gap_source["patchcore_for_gap"] - gap_source["surrogate_for_gap"]
)

display(method_summary.round(6))

method_summary_csv = TABLE_DIR / "mvtec_cf_confirmation_method_average_metrics.csv"
method_summary_latex = TABLE_DIR / "mvtec_cf_confirmation_method_average_metrics.tex"

method_summary.to_csv(method_summary_csv, index=False)
method_summary.round(6).to_latex(method_summary_latex, index=False, escape=True)

print("Saved method-average metrics:", method_summary_csv)
print("Saved method-average LaTeX:", method_summary_latex)

fig, ax = plt.subplots(figsize=SINGLE_COL)

x = np.arange(len(method_summary))
width = 0.24

plot_specs = [
    ("cf_method_score", "CF-selected score", "#1f77b4"),
    ("surrogate_pred", "Surrogate F1", "#ff7f0e"),
    ("f1_test", "Confirmed PatchCore F1", "#2ca02c"),
]

for j, (col, label, color) in enumerate(plot_specs):
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"

    offset = (j - 1) * width

    heights = method_summary[mean_col].astype(float).to_numpy()
    errors = method_summary[std_col].astype(float).to_numpy()

    ax.bar(
        x + offset,
        heights,
        width=width,
        yerr=errors,
        capsize=2,
        label=label,
        color=color,
        linewidth=0.6,
    )

# Annotate the main confirmation gap: Confirmed PatchCore F1 minus Surrogate F1.
for i, row in method_summary.iterrows():
    if pd.notna(row["test_minus_surrogate"]):
        y_text = max(
            float(gap_source.loc[i, "surrogate_for_gap"]),
            float(gap_source.loc[i, "patchcore_for_gap"]),
        ) + 0.015

        ax.text(
            i,
            y_text,
            f"gap {row['test_minus_surrogate']:+.3f}",
            ha="center",
            va="bottom",
            fontsize=7,
        )

ax.set_ylabel("F1 score")
ax.set_xlabel("Counterfactual generation method", labelpad=4)
ax.set_xticks(x)
ax.set_xticklabels(method_summary["method"])

y_values = []
for col in ["cf_method_score_mean", "surrogate_pred_mean", "f1_test_mean"]:
    y_values.extend(method_summary[col].dropna().astype(float).tolist())

if y_values:
    y_min = max(0.0, min(y_values) - 0.08)
    y_max = min(1.0, max(y_values) + 0.08)
    ax.set_ylim(y_min, y_max)

ax.grid(axis="y", linewidth=0.4, alpha=0.35)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=3,
    columnspacing=0.9,
    handlelength=1.3,
    handletextpad=0.4,
)

fig.subplots_adjust(bottom=0.34)

save_paper_fig(fig, "mvtec_cf_confirmation_method_average_method_surrogate_patchcore")

# -------- Plot 2: surrogate-vs-test agreement --------

fig, ax = plt.subplots(figsize=SINGLE_COL)

xs_all = paper_df["surrogate_pred"].astype(float)
ys_all = paper_df["f1_test"].astype(float)

lo = max(0.0, min(xs_all.min(), ys_all.min()) - 0.03)
hi = min(1.0, max(xs_all.max(), ys_all.max()) + 0.03)

ax.plot(
    [lo, hi],
    [lo, hi],
    linestyle="--",
    linewidth=0.9,
    label="Ideal agreement",
)

for method, group in paper_df.groupby("method"):
    ax.scatter(
        group["surrogate_pred"].astype(float),
        group["f1_test"].astype(float),
        s=32,
        alpha=0.8,
        label=f"{method} (n={len(group)})",
    )

best_idx = paper_df["f1_test"].astype(float).idxmax()
largest_gap_idx = paper_df["gap_test_minus_surrogate_pred"].abs().idxmax()

for idx, label in [(best_idx, "best"), (largest_gap_idx, "largest gap")]:
    row = paper_df.loc[idx]
    ax.annotate(
        label,
        (
            float(row["surrogate_pred"]),
            float(row["f1_test"]),
        ),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=7,
    )

ax.set_xlabel("Surrogate F1")
ax.set_ylabel("Confirmed PatchCore F1")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.legend(frameon=False)
ax.grid(linewidth=0.4, alpha=0.35)

save_paper_fig(fig, "mvtec_cf_confirmation_surrogate_vs_test_clean")

# -------- Plot 3: confirmed real F1 distribution by method --------

method_counts = paper_df["method"].value_counts()

if method_counts.size > 1:
    fig, ax = plt.subplots(figsize=SINGLE_COL)

    methods = available_methods
    rng = np.random.default_rng(SEED)

    for i, method in enumerate(methods):
        vals = paper_df.loc[paper_df["method"] == method, "f1_test"].astype(float).values

        if len(vals) == 0:
            continue

        jitter = rng.normal(loc=0.0, scale=0.035, size=len(vals))

        ax.scatter(
            np.full(len(vals), i) + jitter,
            vals,
            s=24,
            alpha=0.75,
        )

        ax.plot(
            i,
            vals.mean(),
            marker="D",
            markersize=5,
        )

    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels(methods)
    ax.set_ylabel("Confirmed PatchCore F1")
    ax.set_xlabel("Counterfactual generation method")
    ax.grid(axis="y", linewidth=0.4, alpha=0.35)

    save_paper_fig(fig, "mvtec_cf_confirmation_method_comparison")
else:
    print("Skipped method-comparison plot because only one method is present:", method_counts.to_dict())

print("\nSaved paper figures:")
for p in saved_figures:
    print("-", p)

Paper table:


,Candidate,Method,CF id,CF method score,Surrogate F1,Real val F1,Confirmed PatchCore F1,Test AUROC,Surrogate - method,Test - method,Test - surrogate,Review budget,Training fraction,Corruption,Backbone,Image size,Layers,Runtime (s)
0,ACFX-21,ACFX,21,NaN,0.781,0.611,0.625,0.487,NaN,NaN,-0.156,0.495,0.73,mild,resnet18,224,l2,28.750
1,ACFX-16,ACFX,16,NaN,0.795,0.582,0.621,0.479,NaN,NaN,-0.174,0.495,0.89,mild,resnet18,224,l2,33.659
2,ACFX-18,ACFX,18,NaN,0.791,0.589,0.621,0.478,NaN,NaN,-0.170,0.495,0.84,mild,resnet18,224,l2,33.096
3,ACFX-19,ACFX,19,NaN,0.791,0.589,0.621,0.478,NaN,NaN,-0.169,0.495,0.75,mild,resnet18,224,l2,29.543
4,ACFX-17,ACFX,17,NaN,0.794,0.589,0.619,0.477,NaN,NaN,-0.175,0.495,0.74,mild,resnet18,224,l2,28.775
5,ACFX-20,ACFX,20,NaN,0.787,0.589,0.602,0.476,NaN,NaN,-0.185,0.495,0.79,mild,resnet18,224,l2,30.395
6,ACFX-23,ACFX,23,NaN,0.740,0.584,0.580,0.466,NaN,NaN,-0.160,0.480,0.64,mild,resnet18,224,l2,26.108
7,ACFX-22,ACFX,22,NaN,0.753,0.554,0.564,0.451,NaN,NaN,-0.189,0.495,0.89,mild,resnet18,224,l2,32.592
8,CFNOW-8,CFNOW,8,0.820,0.823,0.559,0.575,0.536,0.003,-0.245,-0.247,0.500,0.77,none,wide_resnet50_2,256,l3,18.975
9,CFNOW-11,CFNOW,11,0.820,0.819,0.568,0.568,0.534,-0.001,-0.252,-0.250,0.495,0.81,none,wide_resnet50_2,256,l3,19.431


Saved paper table CSV: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_paper_table.csv
Saved paper table LaTeX: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_paper_table.tex


,method,n,cf_method_score_mean,cf_method_score_std,surrogate_pred_mean,surrogate_pred_std,f1_test_mean,f1_test_std,f1_val_real_mean,f1_val_real_std,test_minus_surrogate
0,ACFX,8,0.778851,0.020750,NaN,NaN,0.606696,0.022866,0.586112,0.015433,-0.172155
1,CFNOW,8,0.814939,0.005719,0.814982,0.005652,0.568671,0.002872,0.562176,0.003948,-0.246311
2,DiCE,8,0.773207,0.011409,0.773207,0.011409,0.622871,0.016719,0.611435,0.014017,-0.150336


Saved method-average metrics: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_method_average_metrics.csv
Saved method-average LaTeX: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_method_average_metrics.tex

Saved paper figures:
- /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_cf_confirmation_method_average_method_surrogate_patchcore.pdf
- /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_cf_confirmation_method_average_method_surrogate_patchcore.svg
- /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_cf_confirmation_method_average_method_surrogate_patchcore.png
- /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_cf_confirmation_surrogate_vs_test_clean.pdf
- /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_eval

In [38]:
# ============================================================
# 12) Save evaluation outputs and metadata
# ============================================================

stem = "mvtec_all_methods_cfs" if CFS_FILENAME == ALL_METHODS_CFS_FILENAME else "mvtec_dice_cfs"

eval_csv_path = EVAL_DIR / f"{stem}_patchcore_evaluation.csv"
eval_df.to_csv(eval_csv_path, index=False)

summary_cols = [
    "cf_id",
    "method",
    "method_rank",
    "status",
    "cf_method_score",
    "cf_table_value",
    "surrogate_pred_file",
    "surrogate_pred_fresh",
    "surrogate_pred",
    "f1_val_real",
    "f1_test",
    "auroc_test",
    "gap_fresh_surrogate_minus_method_score",
    "gap_test_minus_method_score",
    "gap_test_minus_surrogate_pred",
    "runtime_sec",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
]

summary_cols = [c for c in summary_cols if c in eval_df.columns]

summary_csv_path = EVAL_DIR / f"{stem}_patchcore_evaluation_summary.csv"
eval_df[summary_cols].to_csv(summary_csv_path, index=False)

metadata_out = {
    "dataset": "mvtec_4class",
    "evaluation_type": "real_patchcore_confirmation",
    "root": str(ROOT),
    "cfs_filename": CFS_FILENAME,
    "cfs_path": str(CFS_PATH),
    "cfs_metadata_path": str(CFS_META_PATH),
    "surrogate_model_path": str(MODEL_PATH),
    "surrogate_metadata_path": str(SURROGATE_META_PATH),
    "train_path": str(TRAIN_PATH),
    "val_path": str(VAL_PATH),
    "test_path": str(TEST_PATH),
    "device": DEVICE,
    "seed": SEED,
    "n_cfs_loaded": int(len(cfs)),
    "n_cfs_evaluated": int(len(eval_df)),
    "n_successful": int((eval_df["status"] == "ok").sum()) if "status" in eval_df.columns else None,
    "features": FEATURES,
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "target": TARGET,
    "score_semantics": {
        "cf_method_score": "Score returned/preserved by the CF generation method, if available.",
        "surrogate_pred": "Fresh CatBoost surrogate prediction computed in the evaluation notebook.",
        "f1_test": "Real confirmed PatchCore test F1.",
    },
}

if not ok_df.empty:
    metadata_out.update({
        "method_counts": ok_df["method"].value_counts().to_dict() if "method" in ok_df.columns else None,
        "mean_cf_method_score": float(ok_df["cf_method_score"].mean()) if ok_df["cf_method_score"].notna().any() else None,
        "mean_surrogate_pred": float(ok_df["surrogate_pred"].mean()),
        "mean_f1_test": float(ok_df["f1_test"].mean()),
        "median_f1_test": float(ok_df["f1_test"].median()),
        "mean_auroc_test": float(ok_df["auroc_test"].mean()),
        "mean_gap_fresh_surrogate_minus_method_score": float(ok_df["gap_fresh_surrogate_minus_method_score"].mean()) if ok_df["gap_fresh_surrogate_minus_method_score"].notna().any() else None,
        "mean_gap_test_minus_method_score": float(ok_df["gap_test_minus_method_score"].mean()) if ok_df["gap_test_minus_method_score"].notna().any() else None,
        "mean_gap_test_minus_surrogate_pred": float(ok_df["gap_test_minus_surrogate_pred"].mean()),
        "mean_runtime_sec": float(ok_df["runtime_sec"].mean()),
    })

if "paper_table_csv" in globals():
    metadata_out["paper_table_csv"] = str(paper_table_csv)
if "paper_table_latex" in globals():
    metadata_out["paper_table_latex"] = str(paper_table_latex)
if "method_summary_csv" in globals():
    metadata_out["method_summary_csv"] = str(method_summary_csv)
if "method_summary_latex" in globals():
    metadata_out["method_summary_latex"] = str(method_summary_latex)
if "saved_figures" in globals():
    metadata_out["saved_figures"] = [str(p) for p in saved_figures]

metadata_json_path = EVAL_DIR / f"{stem}_patchcore_evaluation_metadata.json"

with open(metadata_json_path, "w", encoding="utf-8") as f:
    json.dump(metadata_out, f, indent=2)

print("Saved evaluation table:", eval_csv_path)
print("Saved compact summary:", summary_csv_path)
print("Saved metadata:", metadata_json_path)

print("\nMetadata:")
print(json.dumps(metadata_out, indent=2))

Saved evaluation table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation.csv
Saved compact summary: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation_summary.csv
Saved metadata: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation_metadata.json

Metadata:
{
  "dataset": "mvtec_4class",
  "evaluation_type": "real_patchcore_confirmation",
  "root": "/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class",
  "cfs_filename": "mvtec_all_methods_cfs.csv",
  "cfs_path": "/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals/mvtec_all_methods_cfs.csv",
  "cfs_metadata_path": "/home/jovyan/fast/x_peer_pdt_hyperoptx

In [39]:
# ============================================================
# 13) Final summary for paper
# ============================================================

print("=== MVTec counterfactual evaluation summary ===")
print("CF source file:", CFS_FILENAME)
print("CFs loaded:", len(cfs))
print("CFs evaluated:", len(eval_df))
print("Successful evaluations:", len(ok_df))

if not ok_df.empty:
    if ok_df["cf_method_score"].notna().any():
        print(f"Mean CF-method score:       {ok_df['cf_method_score'].mean():.6f}")
    else:
        print("Mean CF-method score:       n/a")

    print(f"Mean fresh surrogate F1:    {ok_df['surrogate_pred'].mean():.6f}")
    print(f"Mean real validation F1:    {ok_df['f1_val_real'].mean():.6f}")
    print(f"Mean real test F1:          {ok_df['f1_test'].mean():.6f}")
    print(f"Mean test AUROC:            {ok_df['auroc_test'].mean():.6f}")

    if ok_df["gap_fresh_surrogate_minus_method_score"].notna().any():
        print(
            "Mean surrogate-method gap:  "
            f"{ok_df['gap_fresh_surrogate_minus_method_score'].mean():.6f}"
        )

    print(f"Mean test-surrogate gap:    {ok_df['gap_test_minus_surrogate_pred'].mean():.6f}")
    print(f"Mean runtime:               {ok_df['runtime_sec'].mean():.2f} seconds")

    if "method" in ok_df.columns:
        print("\nSuccessful evaluations by method:")
        display(ok_df["method"].value_counts())

    print("\nBest confirmed CF by real test F1:")
    best = ok_df.sort_values("f1_test", ascending=False).iloc[0]
    display(best.to_frame("value"))

if "method_summary" in globals():
    print("\nMethod-average paper summary:")
    display(method_summary.round(6))

if "paper_table" in globals():
    print("\nPaper table preview:")
    display(paper_table)

print("\nMain outputs:")
print("- Evaluation table:", eval_csv_path)
print("- Summary table:", summary_csv_path)
print("- Metadata:", metadata_json_path)

if "paper_table_csv" in globals():
    print("- Paper table CSV:", paper_table_csv)
if "paper_table_latex" in globals():
    print("- Paper table LaTeX:", paper_table_latex)
if "method_summary_csv" in globals():
    print("- Method-average CSV:", method_summary_csv)
if "method_summary_latex" in globals():
    print("- Method-average LaTeX:", method_summary_latex)

if "saved_figures" in globals():
    print("\nPaper figures:")
    for p in saved_figures:
        print("-", p)

=== MVTec counterfactual evaluation summary ===
CF source file: mvtec_all_methods_cfs.csv
CFs loaded: 24
CFs evaluated: 24
Successful evaluations: 24
Mean CF-method score:       0.794073
Mean fresh surrogate F1:    0.789014
Mean real validation F1:    0.586574
Mean real test F1:          0.599413
Mean test AUROC:            0.495678
Mean surrogate-method gap:  0.000021
Mean test-surrogate gap:    -0.189601
Mean runtime:               30.76 seconds

Successful evaluations by method:


DiCE     8
ACFX     8
CFNOW    8
Name: method, dtype: int64


Best confirmed CF by real test F1:


,value
cf_id,1
method,DiCE
method_rank,1
global_cf_id,1
strictly_valid,True
n_violations,0
distance_to_factual,1.452525
generation_sec,0.3643
cf_method_score,0.782018
cf_table_value,0.782018



Method-average paper summary:


,method,n,cf_method_score_mean,cf_method_score_std,surrogate_pred_mean,surrogate_pred_std,f1_test_mean,f1_test_std,f1_val_real_mean,f1_val_real_std,test_minus_surrogate
0,ACFX,8,0.778851,0.020750,NaN,NaN,0.606696,0.022866,0.586112,0.015433,-0.172155
1,CFNOW,8,0.814939,0.005719,0.814982,0.005652,0.568671,0.002872,0.562176,0.003948,-0.246311
2,DiCE,8,0.773207,0.011409,0.773207,0.011409,0.622871,0.016719,0.611435,0.014017,-0.150336



Paper table preview:


,Candidate,Method,CF id,CF method score,Surrogate F1,Real val F1,Confirmed PatchCore F1,Test AUROC,Surrogate - method,Test - method,Test - surrogate,Review budget,Training fraction,Corruption,Backbone,Image size,Layers,Runtime (s)
0,ACFX-21,ACFX,21,NaN,0.781,0.611,0.625,0.487,NaN,NaN,-0.156,0.495,0.73,mild,resnet18,224,l2,28.750
1,ACFX-16,ACFX,16,NaN,0.795,0.582,0.621,0.479,NaN,NaN,-0.174,0.495,0.89,mild,resnet18,224,l2,33.659
2,ACFX-18,ACFX,18,NaN,0.791,0.589,0.621,0.478,NaN,NaN,-0.170,0.495,0.84,mild,resnet18,224,l2,33.096
3,ACFX-19,ACFX,19,NaN,0.791,0.589,0.621,0.478,NaN,NaN,-0.169,0.495,0.75,mild,resnet18,224,l2,29.543
4,ACFX-17,ACFX,17,NaN,0.794,0.589,0.619,0.477,NaN,NaN,-0.175,0.495,0.74,mild,resnet18,224,l2,28.775
5,ACFX-20,ACFX,20,NaN,0.787,0.589,0.602,0.476,NaN,NaN,-0.185,0.495,0.79,mild,resnet18,224,l2,30.395
6,ACFX-23,ACFX,23,NaN,0.740,0.584,0.580,0.466,NaN,NaN,-0.160,0.480,0.64,mild,resnet18,224,l2,26.108
7,ACFX-22,ACFX,22,NaN,0.753,0.554,0.564,0.451,NaN,NaN,-0.189,0.495,0.89,mild,resnet18,224,l2,32.592
8,CFNOW-8,CFNOW,8,0.820,0.823,0.559,0.575,0.536,0.003,-0.245,-0.247,0.500,0.77,none,wide_resnet50_2,256,l3,18.975
9,CFNOW-11,CFNOW,11,0.820,0.819,0.568,0.568,0.534,-0.001,-0.252,-0.250,0.495,0.81,none,wide_resnet50_2,256,l3,19.431



Main outputs:
- Evaluation table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation.csv
- Summary table: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation_summary.csv
- Metadata: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals_evaluation/mvtec_all_methods_cfs_patchcore_evaluation_metadata.json
- Paper table CSV: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_paper_table.csv
- Paper table LaTeX: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_paper_table.tex
- Method-average CSV: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_cf_confirmation_method_average_metrics.

In [40]:
# -------- Plot: soft-parameter profile vs confirmed PatchCore F1 --------
# This plot studies the result space, not the CF methods.
# It shows how confirmed PatchCore F1 varies with human-controllable soft parameters.

soft_df = paper_df.copy()

soft_df["train_fraction"] = pd.to_numeric(
    soft_df["params_soft_train_fraction"],
    errors="coerce",
)

soft_df["review_budget"] = pd.to_numeric(
    soft_df["params_soft_review_budget"],
    errors="coerce",
)

soft_df["confirmed_patchcore_f1"] = pd.to_numeric(
    soft_df["f1_test"],
    errors="coerce",
)

soft_df["corruption"] = soft_df["params_soft_corruption_level"].astype(str)

soft_df = soft_df.dropna(
    subset=[
        "train_fraction",
        "review_budget",
        "confirmed_patchcore_f1",
        "corruption",
    ]
).copy()

display(
    soft_df[
        [
            "cf_id",
            "method",
            "train_fraction",
            "review_budget",
            "corruption",
            "surrogate_pred",
            "confirmed_patchcore_f1",
        ]
    ].sort_values("confirmed_patchcore_f1", ascending=False).round(6)
)

soft_summary = (
    soft_df
    .groupby("corruption")
    .agg(
        n=("cf_id", "count"),
        mean_train_fraction=("train_fraction", "mean"),
        mean_review_budget=("review_budget", "mean"),
        mean_confirmed_patchcore_f1=("confirmed_patchcore_f1", "mean"),
        std_confirmed_patchcore_f1=("confirmed_patchcore_f1", "std"),
    )
    .reset_index()
)

display(soft_summary.round(6))

soft_summary_csv = TABLE_DIR / "mvtec_soft_parameter_summary.csv"
soft_summary.to_csv(soft_summary_csv, index=False)
print("Saved soft-parameter summary:", soft_summary_csv)

fig, ax = plt.subplots(figsize=SINGLE_COL)

corruption_levels = sorted(soft_df["corruption"].unique())
markers = ["o", "s", "^", "D", "v", "P"]

for marker, corruption in zip(markers, corruption_levels):
    group = soft_df[soft_df["corruption"] == corruption].copy()

    ax.scatter(
        group["train_fraction"],
        group["confirmed_patchcore_f1"],
        s=36,
        marker=marker,
        alpha=0.8,
        label=f"Corruption: {corruption}",
    )

# Add simple trend line if enough variation exists.
x_vals = soft_df["train_fraction"].astype(float).to_numpy()
y_vals = soft_df["confirmed_patchcore_f1"].astype(float).to_numpy()

if len(np.unique(x_vals)) >= 2:
    coef = np.polyfit(x_vals, y_vals, deg=1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    y_line = coef[0] * x_line + coef[1]

    ax.plot(
        x_line,
        y_line,
        linestyle="--",
        linewidth=0.9,
        label="Linear trend",
    )

ax.set_xlabel("Training fraction")
ax.set_ylabel("Confirmed PatchCore F1")
ax.grid(linewidth=0.4, alpha=0.35)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.24),
    ncol=2,
    columnspacing=0.9,
    handlelength=1.3,
    handletextpad=0.4,
)

fig.subplots_adjust(bottom=0.32)

save_paper_fig(fig, "mvtec_soft_train_fraction_vs_patchcore_f1")

,cf_id,method,train_fraction,review_budget,corruption,surrogate_pred,confirmed_patchcore_f1
16,1,DiCE,0.75,0.494,mild,0.782018,0.638889
18,4,DiCE,0.75,0.494,mild,0.768456,0.638889
17,3,DiCE,0.75,0.489,mild,0.773289,0.638889
19,2,DiCE,0.75,0.496,mild,0.777679,0.629371
0,21,ACFX,0.73,0.495,mild,0.780857,0.625000
20,6,DiCE,0.75,0.486,none,0.762813,0.624561
2,18,ACFX,0.84,0.495,mild,0.790701,0.621160
3,19,ACFX,0.75,0.495,mild,0.790506,0.621160
1,16,ACFX,0.89,0.495,mild,0.794809,0.621160
4,17,ACFX,0.74,0.495,mild,0.794066,0.619048


,corruption,n,mean_train_fraction,mean_review_budget,mean_confirmed_patchcore_f1,std_confirmed_patchcore_f1
0,mild,15,0.744000,0.492000,0.614132,0.021648
1,none,9,0.822222,0.496222,0.574881,0.018823


Saved soft-parameter summary: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_soft_parameter_summary.csv


[PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_train_fraction_vs_patchcore_f1.pdf'),
 PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_train_fraction_vs_patchcore_f1.svg'),
 PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_train_fraction_vs_patchcore_f1.png')]

In [41]:
# -------- Plot: soft-parameter profile of top confirmed candidates --------
# This plot is about the result space, not the CF methods.
# It compares the human-controllable soft parameters of the best confirmed
# PatchCore candidates against the remaining candidates.

soft_profile_df = paper_df.copy()

soft_profile_df["train_fraction"] = pd.to_numeric(
    soft_profile_df["params_soft_train_fraction"],
    errors="coerce",
)

soft_profile_df["review_budget"] = pd.to_numeric(
    soft_profile_df["params_soft_review_budget"],
    errors="coerce",
)

soft_profile_df["confirmed_patchcore_f1"] = pd.to_numeric(
    soft_profile_df["f1_test"],
    errors="coerce",
)

soft_profile_df["corruption_level"] = (
    soft_profile_df["params_soft_corruption_level"]
    .astype(str)
)

soft_profile_df = soft_profile_df.dropna(
    subset=[
        "train_fraction",
        "review_budget",
        "confirmed_patchcore_f1",
        "corruption_level",
    ]
).copy()

# Define the top confirmed candidates.
# With 24 candidates, this selects the top 6.
top_k = max(3, int(np.ceil(0.25 * len(soft_profile_df))))

top_ids = (
    soft_profile_df
    .sort_values("confirmed_patchcore_f1", ascending=False)
    .head(top_k)
    .index
)

soft_profile_df["confirmed_group"] = "Other candidates"
soft_profile_df.loc[top_ids, "confirmed_group"] = "Top confirmed candidates"

soft_profile_df["confirmed_group"] = pd.Categorical(
    soft_profile_df["confirmed_group"],
    categories=["Top confirmed candidates", "Other candidates"],
    ordered=True,
)

soft_profile_summary = (
    soft_profile_df
    .groupby("confirmed_group")
    .agg(
        n=("cf_id", "count"),
        mean_confirmed_patchcore_f1=("confirmed_patchcore_f1", "mean"),
        mean_train_fraction=("train_fraction", "mean"),
        mean_review_budget=("review_budget", "mean"),
    )
    .reset_index()
)

display(soft_profile_summary.round(6))

soft_profile_csv = TABLE_DIR / "mvtec_soft_parameter_profile_top_confirmed.csv"
soft_profile_df.to_csv(soft_profile_csv, index=False)
print("Saved soft-parameter profile:", soft_profile_csv)

# Prepare data for boxplots.
group_order = ["Top confirmed candidates", "Other candidates"]

train_data = [
    soft_profile_df.loc[
        soft_profile_df["confirmed_group"] == group,
        "train_fraction",
    ].astype(float).values
    for group in group_order
]

budget_data = [
    soft_profile_df.loc[
        soft_profile_df["confirmed_group"] == group,
        "review_budget",
    ].astype(float).values
    for group in group_order
]

# Prepare corruption proportions.
corruption_counts = (
    soft_profile_df
    .groupby(["confirmed_group", "corruption_level"])
    .size()
    .unstack(fill_value=0)
    .reindex(group_order)
)

corruption_props = corruption_counts.div(
    corruption_counts.sum(axis=1),
    axis=0,
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(6.85, 2.45),
    gridspec_kw={"width_ratios": [1.0, 1.0, 1.2]},
)

# Panel A: training fraction
axes[0].boxplot(
    train_data,
    labels=["Top", "Other"],
    widths=0.55,
    showmeans=True,
)

axes[0].set_ylabel("Training fraction")
axes[0].set_xlabel("Confirmed group")
axes[0].grid(axis="y", linewidth=0.4, alpha=0.35)

# Panel B: review budget
axes[1].boxplot(
    budget_data,
    labels=["Top", "Other"],
    widths=0.55,
    showmeans=True,
)

axes[1].set_ylabel("Review budget")
axes[1].set_xlabel("Confirmed group")
axes[1].grid(axis="y", linewidth=0.4, alpha=0.35)

# Panel C: corruption-level composition
bottom = np.zeros(len(corruption_props))

x_pos = np.arange(len(corruption_props))

for corruption in corruption_props.columns:
    values = corruption_props[corruption].astype(float).values

    axes[2].bar(
        x_pos,
        values,
        bottom=bottom,
        width=0.55,
        label=str(corruption),
    )

    bottom += values

axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(["Top", "Other"])
axes[2].set_ylim(0, 1.0)
axes[2].set_ylabel("Proportion")
axes[2].set_xlabel("Confirmed group")
axes[2].grid(axis="y", linewidth=0.4, alpha=0.35)

axes[2].legend(
    title="Corruption",
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=max(1, len(corruption_props.columns)),
    fontsize=7,
    title_fontsize=7,
)

# Add n labels.
for ax_i, group in enumerate(group_order):
    n_group = int((soft_profile_df["confirmed_group"] == group).sum())

    axes[0].text(
        ax_i + 1,
        axes[0].get_ylim()[1],
        f"n={n_group}",
        ha="center",
        va="bottom",
        fontsize=7,
    )

    axes[1].text(
        ax_i + 1,
        axes[1].get_ylim()[1],
        f"n={n_group}",
        ha="center",
        va="bottom",
        fontsize=7,
    )

fig.subplots_adjust(bottom=0.32, wspace=0.45)

save_paper_fig(fig, "mvtec_soft_parameter_profile_top_confirmed")

,confirmed_group,n,mean_confirmed_patchcore_f1,mean_train_fraction,mean_review_budget
0,Top confirmed candidates,6,0.632600,0.746667,0.492333
1,Other candidates,18,0.588351,0.782222,0.494000


Saved soft-parameter profile: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/tables_cfs_evaluation/mvtec_soft_parameter_profile_top_confirmed.csv


[PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_parameter_profile_top_confirmed.pdf'),
 PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_parameter_profile_top_confirmed.svg'),
 PosixPath('/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/figures_cfs_evaluation/mvtec_soft_parameter_profile_top_confirmed.png')]